# Data Cleansing


## Author: 
Yangshunshun Lu  

## AI Disclosure Statement:
This code uses Google Gemini for debugging and for learning the `langdetect` and `chardet` packages.

## Description:
This script implements a comprehensive data cleansing pipeline that merges multiple datasets, performs language detection to retain English-only tracks, and executes deep text cleaning and lemmatization to prepare lyrics for textual analysis and EDA.

In [ ]:
# ==========================================
# Data Cleansing & Preprocessing Pipeline
# ==========================================
import os
import glob
import pandas as pd
import numpy as np
import spacy
from langdetect import detect, DetectorFactory
import chardet

# Ensure consistent language detection
DetectorFactory.seed = 0

# Set environment variables for encoding
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["PYTHONUTF8"] = "1"

print('Libraries imported successfully.')

In [2]:
#! pip install chardet

## Merge European + US' lyrics data and Basic Cleaning


In [ ]:
# ==========================================
# Step 1: Memory-Efficient Merge & Clean
# Merge CSVs, clean missing values, and align columns in one pass
# ==========================================
INPUT_DIR = '../../Data/match_output_features'
CLEANED_MERGE_FILE = '../../Data/cleaned_data/lyricsFeatures_mergeERUS_clean.csv'

os.makedirs(os.path.dirname(CLEANED_MERGE_FILE), exist_ok=True)
if os.path.exists(CLEANED_MERGE_FILE):
    os.remove(CLEANED_MERGE_FILE)
    print(f"Deleted old merged file: {CLEANED_MERGE_FILE}")

TARGET_COUNTRIES = [
    "Germany", "France", "United Kingdom", "Italy", "Spain", 
    "Netherlands", "Belgium", "Sweden", "Norway", "Denmark", 
    "Finland", "Ireland", "Austria", "Switzerland", "Portugal", 
    "Poland", "Czech Republic", "USA"
]

AUDIO_FEATURES = [
    'danceability', 'energy', 'loudness', 'speechiness', 
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]

def smart_convert_datetime(val):
    """
    Converts various input formats into pandas datetime objects with heuristic handling.
    
    This function attempts to intelligently parse date information from strings, 
    null values, or numeric timestamps. It specifically checks if a numeric value 
    represents a Unix timestamp in milliseconds (values > 1e9) and converts it 
    accordingly. For other formats, it relies on standard pandas datetime coercion.

    Args:
        val (str, int, float, or scalar): The value to be converted into a 
            datetime object.

    Returns:
        pd.Timestamp: A converted pandas Timestamp object. Returns pd.NaT 
            if the input is null, an empty string, or an unparseable format.
    """
    if pd.isna(val) or val == "": return pd.NaT
    try:
        num_val = pd.to_numeric(val, errors='raise')
        if num_val > 1e9: return pd.to_datetime(num_val, unit='ms')
    except: pass
    return pd.to_datetime(val, errors='coerce')

all_files = glob.glob(os.path.join(INPUT_DIR, "*.csv"))
is_first_file = True

print(f"Starting Merge & Clean for {len(TARGET_COUNTRIES)} countries...")

for file_path in all_files:
    file_name = os.path.basename(file_path)
    country_name = next((c for c in TARGET_COUNTRIES if file_name.startswith(c)), None)
    if not country_name: continue

    # Detect Encoding
    with open(file_path, 'rb') as f:
        encoding = chardet.detect(f.read(100000))['encoding'] or 'utf-8'

    try:
        df = pd.read_csv(file_path, encoding=encoding, low_memory=False)
        
        # 1. Basic Cleaning
        df = df.dropna(axis=1, how='all')
        df = df.loc[:, ~df.columns.str.contains('^Unnamed')]
        df['country'] = country_name
        
        # 2. Filter valid lyrics
        if 'lyrics' in df.columns:
            df['lyrics'] = df['lyrics'].astype(str).str.lower()
            df = df[~df['lyrics'].str.contains('no lyrics available', case=False, na=False)]
            df = df[df['lyrics'].str.len() > 10]
        else:
            continue
            
        # 3. Filter valid audio features
        existing_features = [col for col in AUDIO_FEATURES if col in df.columns]
        if existing_features:
            for col in existing_features:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            df = df.dropna(subset=existing_features)
            
        # 4. Process dates & duplicates
        if 'fetched_at' in df.columns:
            df['date'] = df['fetched_at'].apply(smart_convert_datetime)
        
        df = df.drop_duplicates(subset=['uri'])
        
        # 5. Append to master file
        df.to_csv(CLEANED_MERGE_FILE, mode='a', index=False, encoding='utf-8-sig', header=is_first_file)
        
        print(f"Processed & Appended: {country_name} ({len(df)} rows)")
        is_first_file = False
        del df
        
    except Exception as e:
        print(f"Failed to process {country_name}: {e}")

print(f"\n✨ Step 1 Complete! Cleaned data saved to: {CLEANED_MERGE_FILE}")

## Remove non English Lyrics

In [ ]:
# ==========================================
# Step 2: Strict English Language Detection
# Check the middle of the lyrics to avoid misclassifying foreign songs with English intros
# ==========================================
print("Loading cleaned data for language filtering...")
df = pd.read_csv(CLEANED_MERGE_FILE, low_memory=False)
initial_count = len(df)

def strict_english_check(text):
    """
    Performs a localized language check to verify if the text is primarily English.
    
    This function implements a strict verification by first ensuring the text 
    meets a minimum length requirement. To avoid false positives from non-English 
    intros or metadata, it extracts a 300-character sample from the middle of 
    the text and uses the `detect` library to identify the language.

    Args:
        text (str): The raw text or lyrics to be validated.

    Returns:
        bool: True if the middle chunk of the text is detected as English ('en') 
              and meets the length threshold; False otherwise.
    """
    text = str(text)
    if len(text) < 50: return False
    
    # Extract middle 300 characters to bypass intro
    mid = len(text) // 2
    chunk = text[mid-150 : mid+150]
    try:
        return detect(chunk) == 'en'
    except:
        return False

print("Performing strict English detection (this may take a few minutes)...")
df['is_english'] = df['lyrics'].apply(strict_english_check)

# Keep only pure English songs
en_df = df[df['is_english']].copy()
en_df = en_df.drop(columns=['is_english'])

print("-" * 30)
print(f"Total initial records: {initial_count}")
print(f"Pure English records kept: {len(en_df)}")
print(f"Foreign/Invalid records dropped: {initial_count - len(en_df)}")

## Tokenization
Tokenize the data after cleaning

In [ ]:
# ==========================================
# Step 3: English Tokenization & Final Save
# Since we dropped foreign songs, we only need the English NLP model!
# ==========================================
import re
from nltk.corpus import stopwords

print("Loading English NLP model...")
nlp_en = spacy.load("en_core_web_sm", disable=["parser", "ner"])
stop_words_en = set(stopwords.words('english'))

print("Tokenizing pure English lyrics...")
en_df['processed_lyrics'] = ""

# Pre-clean texts (remove bracketed tags and punctuation)
clean_texts = en_df['lyrics'].str.replace(r'\[.*?\]', ' ', regex=True).str.replace(r'[^\w\s]', ' ', regex=True)

processed = []
# Batch processing for extreme speed
for doc in nlp_en.pipe(clean_texts, batch_size=1000):
    tokens = [t.lemma_ for t in doc if t.text not in stop_words_en and len(t.text) > 1]
    processed.append(" ".join(tokens))

en_df['processed_lyrics'] = processed

# Drop rows where tokenization resulted in empty strings
en_df = en_df[en_df['processed_lyrics'].str.strip() != ""]

# Final Save
FINAL_OUTPUT_FILE = '../Data/cleaned_data/lyricsFeatures_mergeERUS_clean_enonly_token.csv'
en_df.to_csv(FINAL_OUTPUT_FILE, index=False, encoding='utf-8-sig')

print("-" * 30)
print(f"Data Cleansing Pipeline Complete!")
print(f"Final valid English tracks ready for modeling: {len(en_df)}")
print(f"Saved perfectly to: {FINAL_OUTPUT_FILE}")

In [ ]:
#%pip install --only-binary=:all: spacy langdetect nltk